In [1]:
# Step 2: RAG Pipeline
# Install required packages first if needed:
# pip install llama-index chromadb llama-index-vector-stores-chroma

import os
from dotenv import load_dotenv
import json
import chromadb
from llama_index.core import VectorStoreIndex, Document, Settings
from llama_index.vector_stores.chroma import ChromaVectorStore
from llama_index.core import StorageContext

load_dotenv()
print("Libraries loaded!")

Libraries loaded!


In [2]:
# Load enriched data
with open('../data/pubmedqa_entities.json', 'r') as f:
    records = json.load(f)

print(f"Loaded {len(records)} records")
print(f"Sample keys: {list(records[0].keys())}")

Loaded 1000 records
Sample keys: ['pubid', 'question', 'contexts', 'long_answer', 'final_decision', 'entities']


In [3]:
# Create ChromaDB vector store
chroma_client = chromadb.PersistentClient(path="../data/chroma_db")
chroma_collection = chroma_client.get_or_create_collection("pubmedqa")

print(f"ChromaDB collection created: {chroma_collection.name}")

ChromaDB collection created: pubmedqa


In [4]:
# Convert records to LlamaIndex Documents
documents = []
for record in records:
    # Combine all contexts into one text
    full_text = " ".join(record['contexts'])
    
    doc = Document(
        text=full_text,
        metadata={
            'pubid': str(record['pubid']),
            'question': record['question'],
            'final_decision': record['final_decision'],
            'long_answer': record['long_answer']
        }
    )
    documents.append(doc)

print(f"Created {len(documents)} documents")

Created 1000 documents


In [5]:
from llama_index.embeddings.huggingface import HuggingFaceEmbedding
from llama_index.core import VectorStoreIndex, StorageContext
from llama_index.vector_stores.chroma import ChromaVectorStore

# Use HuggingFace embedding model
embed_model = HuggingFaceEmbedding(model_name="BAAI/bge-small-en-v1.5")
Settings.embed_model = embed_model
Settings.llm = None  # No LLM for now

print("Embedding model loaded!")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/94.8k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/743 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/711k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

LLM is explicitly disabled. Using MockLLM.
Embedding model loaded!


In [6]:
# Create vector store and index
vector_store = ChromaVectorStore(chroma_collection=chroma_collection)
storage_context = StorageContext.from_defaults(vector_store=vector_store)

print("Indexing documents... (this may take a few minutes)")
index = VectorStoreIndex.from_documents(
    documents,
    storage_context=storage_context,
    show_progress=True
)

print("Indexing complete!")

Indexing documents... (this may take a few minutes)


Applying transformations:   0%|          | 0/1 [00:00<?, ?it/s]

Generating embeddings:   0%|          | 0/1000 [00:00<?, ?it/s]

Indexing complete!


In [7]:
# Create query engine and test
query_engine = index.as_query_engine(similarity_top_k=3)

# Test query
test_query = "Do mitochondria play a role in programmed cell death?"
response = query_engine.query(test_query)

print("Query:", test_query)
print("\nResponse:", response)

Query: Do mitochondria play a role in programmed cell death?

Response: Context information is below.
---------------------
pubid: 21645374
question: Do mitochondria play a role in remodelling lace plant leaves during programmed cell death?
final_decision: yes
long_answer: Results depicted mitochondrial dynamics in vivo as PCD progresses within the lace plant, and highlight the correlation of this organelle with other organelles during developmental PCD. To the best of our knowledge, this is the first report of mitochondria and chloroplasts moving on transvacuolar strands to form a ring structure surrounding the nucleus during developmental PCD. Also, for the first time, we have shown the feasibility for the use of CsA in a whole plant system. Overall, our findings implicate the mitochondria as playing a critical and early role in developmentally regulated PCD in the lace plant.

Programmed cell death (PCD) is the regulated death of cells within an organism. The lace plant (Aponogeton 

In [14]:
from llama_index.llms.huggingface_api import HuggingFaceInferenceAPI
import os
from dotenv import load_dotenv
load_dotenv()

llm = HuggingFaceInferenceAPI(
    model_name="HuggingFaceH4/zephyr-7b-beta",
    token=os.getenv("HUGGINGFACE_TOKEN")
)

Settings.llm = llm

# Recreate query engine
query_engine = index.as_query_engine(similarity_top_k=3)

test_query = "Do mitochondria play a role in programmed cell death?"
response = query_engine.query(test_query)

print("Query:", test_query)
print("\nResponse:", response.response)

Query: Do mitochondria play a role in programmed cell death?

Response: Empty Response


In [17]:
llm = HuggingFaceInferenceAPI(
    model_name="meta-llama/Llama-3.1-8B-Instruct",
    token=os.getenv("HUGGINGFACE_TOKEN")
)

Settings.llm = llm
test_response = llm.complete("What is programmed cell death? Answer in one sentence.")
print(test_response)

Programmed cell death, also known as apoptosis, is a naturally occurring process in which cells self-destruct and die in a controlled manner, often in response to cellular damage or as part of normal development and tissue maintenance.


In [18]:
# Recreate query engine with working LLM
query_engine = index.as_query_engine(similarity_top_k=3)

test_query = "Do mitochondria play a role in programmed cell death?"
response = query_engine.query(test_query)

print("Query:", test_query)
print("\nResponse:", response.response)

Query: Do mitochondria play a role in programmed cell death?

Response: Yes, mitochondria play a critical and early role in developmentally regulated programmed cell death.


In [19]:
# GraphRAG: Enhance context with Knowledge Graph entities
from neo4j import GraphDatabase

driver = GraphDatabase.driver(
    os.getenv("NEO4J_URI"), 
    auth=(os.getenv("NEO4J_USERNAME"), os.getenv("NEO4J_PASSWORD"))
)

def get_related_entities(query_text, top_k=5):
    """Get related entities from Knowledge Graph"""
    with driver.session() as session:
        result = session.run("""
            MATCH (e:Entity)
            WHERE toLower(e.text) CONTAINS toLower($keyword)
            MATCH (d:Document)-[:HAS_ENTITY]->(e)
            RETURN d.question as question, d.pubid as pubid, 
                   collect(e.text) as entities
            LIMIT $top_k
        """, keyword=query_text.split()[0], top_k=top_k)
        return result.data()

# Test
entities = get_related_entities("mitochondria programmed cell death")
print("Related KG context:")
for r in entities[:3]:
    print(f"  Question: {r['question'][:80]}")
    print(f"  Entities: {r['entities'][:5]}")

Related KG context:


In [20]:
def get_related_entities(query_words, top_k=5):
    """Get related entities from Knowledge Graph"""
    with driver.session() as session:
        result = session.run("""
            MATCH (d:Document)-[:HAS_ENTITY]->(e:Entity)
            RETURN d.question as question, d.pubid as pubid,
                   collect(e.text) as entities
            LIMIT $top_k
        """, top_k=top_k)
        return result.data()

entities = get_related_entities("mitochondria")
print("Related KG context:")
for r in entities[:3]:
    print(f"  Question: {r['question'][:80]}")
    print(f"  Entities: {r['entities'][:5]}")

Related KG context:
  Question: Do mitochondria play a role in remodelling lace plant leaves during programmed c
  Entities: ['Aponogeton', 'PCD', 'MitoTracker Red', 'TUNEL', 'PTP']
  Question: Landolt C and snellen e acuity: differences in strabismus amblyopia?
  Entities: ['Snellen', 'the Landolt C (Precision Vision', 'ETDRS', 'LR']
  Question: Syncope during bathing in infants, a pediatric form of water-induced urticaria?
  Entities: ['aquagenic urticaria']


In [21]:
# GraphRAG: Combine Vector RAG + Knowledge Graph context
def graphrag_query(query_text):
    # Step 1: Get KG context
    kg_context = get_related_entities(query_text)
    kg_text = ""
    for r in kg_context[:3]:
        kg_text += f"Related document: {r['question']}\n"
        kg_text += f"Key entities: {', '.join(r['entities'][:5])}\n\n"
    
    # Step 2: Get vector RAG response
    rag_response = query_engine.query(query_text)
    
    # Step 3: Combine with LLM
    combined_prompt = f"""
Knowledge Graph Context:
{kg_text}

RAG Response: {rag_response.response}

Based on both the knowledge graph context and RAG response, 
answer: {query_text}
"""
    final_response = llm.complete(combined_prompt)
    return final_response

# Test GraphRAG
result = graphrag_query("Do mitochondria play a role in programmed cell death?")
print("GraphRAG Response:")
print(result)

GraphRAG Response:
Based on the knowledge graph context and the RAG response, the answer is:

Yes, mitochondria play a critical and early role in developmentally regulated programmed cell death.

This is supported by the related document "Do mitochondria play a role in remodelling lace plant leaves during programmed cell death?" which suggests that mitochondria are involved in programmed cell death (PCD) in the lace plant.


In [22]:
# Evaluation: Vanilla RAG vs GraphRAG
test_questions = [
    {
        "question": "Do mitochondria play a role in programmed cell death?",
        "expected": "yes"
    },
    {
        "question": "Is the cell death in mesial temporal sclerosis apoptotic?",
        "expected": "maybe"
    },
    {
        "question": "Does aerobic exercise improve cognitive function in older adults?",
        "expected": "yes"
    }
]

results = []

for item in test_questions:
    q = item["question"]
    expected = item["expected"]
    
    # Vanilla RAG
    vanilla_response = query_engine.query(q)
    vanilla_answer = vanilla_response.response.lower()
    
    # GraphRAG
    graph_response = graphrag_query(q)
    graph_answer = str(graph_response).lower()
    
    results.append({
        "question": q[:60],
        "expected": expected,
        "vanilla_rag": vanilla_answer[:100],
        "graphrag": graph_answer[:100]
    })
    print(f"✓ {q[:50]}")

print("\nEvaluation complete!")

✓ Do mitochondria play a role in programmed cell dea
✓ Is the cell death in mesial temporal sclerosis apo
✓ Does aerobic exercise improve cognitive function i

Evaluation complete!


In [23]:
# Display results
print("=" * 80)
print(f"{'QUESTION':<40} {'EXPECTED':<10} {'VANILLA RAG':<15} {'GRAPHRAG'}")
print("=" * 80)

for r in results:
    print(f"\nQ: {r['question']}")
    print(f"Expected: {r['expected']}")
    print(f"Vanilla RAG: {r['vanilla_rag'][:120]}")
    print(f"GraphRAG:    {r['graphrag'][:120]}")
    print("-" * 80)

QUESTION                                 EXPECTED   VANILLA RAG     GRAPHRAG

Q: Do mitochondria play a role in programmed cell death?
Expected: yes
Vanilla RAG: yes, mitochondria play a critical and early role in developmentally regulated programmed cell death.
GraphRAG:    based on the knowledge graph context and the rag response, the answer is:

yes, mitochondria play a 
--------------------------------------------------------------------------------

Q: Is the cell death in mesial temporal sclerosis apoptotic?
Expected: maybe
Vanilla RAG: the data suggest that either apoptosis is not involved in cell loss in mts, or a very slow rate of c
GraphRAG:    based on the rag response, it appears that the cell death in mesial temporal sclerosis is not apopto
--------------------------------------------------------------------------------

Q: Does aerobic exercise improve cognitive function in older ad
Expected: yes
Vanilla RAG: while there isn't direct information about the impact of aerobi